# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umaizaaman/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

if token:
    print("✅ HF_TOKEN loaded successfully!")
else:
    print("❌ HF_TOKEN not found.")

✅ HF_TOKEN loaded successfully!


In [3]:
!pip -q install datasets duckdb pyarrow huggingface_hub

In [4]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

print("✅ Hugging Face login successful!")

✅ Hugging Face login successful!


In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train"
)

print(dataset)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 78835655
})


In [8]:
from datasets import load_dataset
import pandas as pd

sample = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train[:10000]"
)

df = pd.DataFrame(sample)

print(df.shape)
df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

(10000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
## My Rule

I will prioritize content with high impressions, low CTR, and an average position between 5 and 20. These pages already receive visibility in Google Search but are not getting enough clicks, so improving titles and meta descriptions may increase traffic.

### Reason Codes

* **CTR_LOW** – The page has high impressions but a low click-through rate.
* **GOOD_POSITION** – The page is already ranking on the first or second page of Google.
* **HIGH_IMPRESSIONS** – The page has enough search visibility to make optimization worthwhile.

### Action Label

**CTR_FIX** – Improve the title and meta description to increase click-through rate.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: Define baseline rule parameters

IMPRESSION_THRESHOLD = 100
POSITION_MIN = 5
POSITION_MAX = 20

reason_codes = {
    "CTR_LOW": "High impressions but low clicks",
    "GOOD_POSITION": "Average position between 5 and 20",
    "HIGH_IMPRESSIONS": "Page has enough search visibility"
}

ACTION_LABEL = "CTR_FIX"

print("Baseline Rule Parameters")
print("------------------------")
print(f"Impression Threshold : {IMPRESSION_THRESHOLD}")
print(f"Position Range       : {POSITION_MIN} - {POSITION_MAX}")
print(f"Action Label         : {ACTION_LABEL}")
print("\nReason Codes:")
for code, desc in reason_codes.items():
    print(f"- {code}: {desc}")


Baseline Rule Parameters
------------------------
Impression Threshold : 100
Position Range       : 5 - 20
Action Label         : CTR_FIX

Reason Codes:
- CTR_LOW: High impressions but low clicks
- GOOD_POSITION: Average position between 5 and 20
- HIGH_IMPRESSIONS: Page has enough search visibility


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*My baseline score is based on pages with high impressions, low CTR, and a good average position. Pages meeting these conditions receive a higher score because they are visible in search results but are not attracting enough clicks.

Reason Code:
CTR_LOW

Action Label:
CTR_FIX

In [10]:
import os

# Create a simple baseline score
df["baseline_score"] = (
    (df["gsc_impressions"] >= 100).astype(int) +
    ((df["gsc_avg_position"] >= 5) & (df["gsc_avg_position"] <= 20)).astype(int)
)

# Assign reason codes
df["reason_code"] = "CTR_LOW"

df.loc[df["gsc_impressions"] >= 100, "reason_code"] = "HIGH_IMPRESSIONS"
df.loc[
    (df["gsc_avg_position"] >= 5) &
    (df["gsc_avg_position"] <= 20),
    "reason_code"
] = "GOOD_POSITION"

# Action label
df["action_label"] = "CTR_FIX"

# Rank by score and impressions
ranked_df = df.sort_values(
    by=["baseline_score", "gsc_impressions"],
    ascending=[False, False]
)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

ranked_df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

# Show top 10 rows
print(ranked_df[
    [
        "content_hash_id",
        "gsc_impressions",
        "gsc_avg_position",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].head(10))

               content_hash_id  gsc_impressions  gsc_avg_position  \
6854  content_4e8d1e11f60fe6ba              506          7.045455   
9596  content_4e8d1e11f60fe6ba              466          7.500000   
953   content_37b3bafd5f88fdd1              202          5.034653   
9397  content_cf651123f1085418              177          9.988701   
3793  content_6cd0c162158858c3              164          5.524390   
5514  content_6cd0c162158858c3              162          6.024691   
6589  content_5e3f5c78090b9856              160          7.237500   
9042  content_e032cc1f6c7fef57              158          6.670886   
6361  content_e032cc1f6c7fef57              150          6.020000   
6909  content_a64143f6e4a21ffe              139         11.625899   

      baseline_score    reason_code action_label  
6854               2  GOOD_POSITION      CTR_FIX  
9596               2  GOOD_POSITION      CTR_FIX  
953                2  GOOD_POSITION      CTR_FIX  
9397               2  GOOD_POSITION 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
## Top-20 Review

The top-ranked pages have high impressions and good average search positions, making them strong candidates for CTR improvement.

For each of the top 20 pages:

* **Action:** CTR_FIX
* **Reason Code:** GOOD_POSITION
* **Confidence:** Medium to High
* **What would make it wrong?** The page may already have an optimized title and meta description, or the low CTR may be caused by factors outside our dataset, such as strong competitors, search intent mismatch, or rich search results.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Top-20 Review

top20 = ranked_df.head(20)

for i, row in top20.iterrows():
    print(f"Content: {row['content_hash_id']}")
    print(f"Action: {row['action_label']}")
    print(f"Reason Code: {row['reason_code']}")
    print("Confidence: Medium")
    print("What would make it wrong? Better title/meta already exists or search intent differs.")
    print("-" * 60)


Content: content_4e8d1e11f60fe6ba
Action: CTR_FIX
Reason Code: GOOD_POSITION
Confidence: Medium
What would make it wrong? Better title/meta already exists or search intent differs.
------------------------------------------------------------
Content: content_4e8d1e11f60fe6ba
Action: CTR_FIX
Reason Code: GOOD_POSITION
Confidence: Medium
What would make it wrong? Better title/meta already exists or search intent differs.
------------------------------------------------------------
Content: content_37b3bafd5f88fdd1
Action: CTR_FIX
Reason Code: GOOD_POSITION
Confidence: Medium
What would make it wrong? Better title/meta already exists or search intent differs.
------------------------------------------------------------
Content: content_cf651123f1085418
Action: CTR_FIX
Reason Code: GOOD_POSITION
Confidence: Medium
What would make it wrong? Better title/meta already exists or search intent differs.
------------------------------------------------------------
Content: content_6cd0c162158858c

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
## Weak Picks + Leakage Check

Some recommendations may not be correct because high impressions and a good search position do not always mean that a page needs a CTR improvement. Some pages may already have optimized titles and meta descriptions, or low CTR may be caused by user search intent or strong competitors.

Leakage Check:

* No future information was used.
* No product flags or label-derived features were used.
* Only current search performance metrics (impressions and average position) were used to build the baseline score.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Weak Picks + Leakage Check

print("Weak Picks Review")
print("------------------")
print("Some pages may already be optimized despite receiving a high score.")
print("Low CTR can also be caused by search intent or strong competitors.")
print()

print("Leakage Check")
print("-------------")
print("No future-window data used.")
print("No label-derived features used.")
print("Only impressions and average position were used for scoring.")


Weak Picks Review
------------------
Some pages may already be optimized despite receiving a high score.
Low CTR can also be caused by search intent or strong competitors.

Leakage Check
-------------
No future-window data used.
No label-derived features used.
Only impressions and average position were used for scoring.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.